# Mode-classification calibration on GS2 linear databases (Fusion_PhD-7k70, lqwx, use8)

Calibrates the cutoffs of `general_analysis.mode_classification` on every GS2 Latin-hypercube database with per-species fluxes.
Three indicators per case from the final time: tearing parameter `T` (Hatch 2012), signed frequency `omega` (ion direction > 0, pyrokinetics
convention) and `chi_e/chi_i` (Kotschenreuther 2019). Consensus: a label only if every indicator agrees, otherwise `mixed`.
**Fusion_PhD-lqwx (2026-10-01):** the user's final criteria (KBM: T<0.05, omega>0, 0.25<chi_e/chi_i<4; MTM: T>0.15, omega<0).
**Fusion_PhD-use8 (2026-10-05):** indicators now come from each database's cube (`pyroscan.nc`-style `cube.nc` + `pyroscan.json`) through
`mc.indicators(scan, ds)` (`scan.sample_pyro`, Fusion_PhD-k8w8); no run directory is read and no per-run `Pyro` is built. The D_e/chi_e cut and the
7k70 old->new transition table (which needed it) were dropped by the user and are gone with `de_chi`.

In [ ]:
import os, warnings
from multiprocessing import Pool
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dotenv import load_dotenv
from pyrokinetics import Pyro, PyroHypercube
from pyrokinetics.pyroscan import PyroScanGKOutput
from general_analysis import mode_classification as mc

warnings.filterwarnings("ignore")
load_dotenv(Path.cwd().parent / "local.env")
data_root = Path(os.environ["GK_DATA_ROOT"])
code_, run_template, project = "GS2", "Runs", "LATIN_HYPERCUBE"
# database label: (case, scan_information)
# database label: (case, scan_information). Every GS2 LATIN_HYPERCUBE database; load failures are reported.
databases = {"NSTX_MTM n1000": ("NSTX_MTM", "beta_q_shat_ky_n1000"), "NSTX_MTM n300": ("NSTX_MTM", "beta_q_shat_ky_n300"),
             "SPR-045": ("SPR-045", "kbm_8d"), "M1": ("M1", "kbm_8d"), "R4": ("R4", "kbm_8d")}
databases |= {f"SPR-008 {s}": ("SPR-008", s) for s in ("aLTe_shat", "aLTe_q", "aLTe_aLn", "aLTe_aLn_shat", "aLTe_aLn_q_n200",
              "aLTe_aLn_q_n500", "aLTe_aLn_shat_q_n200", "aLTe_aLn_shat_q_n500")}
def load_cube(case, scan_name):
    """(scan, ds) of a GS2 database: the final-time cube where there is one, else the tail-averaged cube."""
    base = data_root / code_ / run_template / project / case / scan_name
    cd = next(base / c for c in ("pyro_cube", "pyro_cube_avg") if (base / c / "cube.nc").exists())
    cube = cd / "cube_eigenfunctions.nc" if (cd / "cube_eigenfunctions.nc").exists() else cd / "cube.nc"  # R4: ragged theta
    scan = PyroHypercube(pyro=Pyro(gk_file=cd / "pyroscan_base.input", gk_code="GS2"), pyroscan_json=cd / "pyroscan.json",
                         base_directory=base, file_name="gs2.in")
    return mc.attach_legacy_funcs(scan), PyroScanGKOutput.from_netcdf(cube).data
n_spot, seed = 5, 0
plot_dir = Path.cwd().parent / "Plots" / "mode_classification_lqwx"; plot_dir.mkdir(parents=True, exist_ok=True)
plt.style.use(Path(mc.__file__).parent / "paper.mplstyle")

In [ ]:
rows, cubes = [], {}
for db, (case, scan_name) in databases.items():
    try:
        scan, ds = load_cube(case, scan_name)
        ind = mc.indicators(scan, ds).to_dataframe()
        cubes[db] = ds
        gamma = ds["growth_rate"].squeeze().pint.dequantify().values
        rows.append(ind.assign(db=db, run=ind.sample_name, growth_rate=gamma, ky=ds["ky"].values.ravel() if "ky" in ds else np.nan))
    except Exception as e:
        rows.append(pd.DataFrame([dict(db=db, run="", err=repr(e)[:120])]))
raw = pd.concat(rows, ignore_index=True)

In [ ]:
# Nothing is dropped silently: load failures, then non-positive growth rate, then non-finite indicators.
failed = raw[raw.get("err").notna()] if "err" in raw else raw.iloc[:0]
ok = raw.drop(failed.index)
failed.to_csv(plot_dir / "load_failures.csv", index=False)
print(failed.groupby("db").err.first().to_dict())
print("loaded", raw.groupby("db").size().to_dict(), "| load failures", failed.groupby("db").size().to_dict())
stable = ok[ok.growth_rate <= 0]
ok = ok[ok.growth_rate > 0]
nonfinite = ok[~np.isfinite(ok[["T", "omega", "chi_ratio"]]).all(axis=1)]
df = ok.drop(nonfinite.index).copy()
print("growth_rate <= 0", stable.groupby("db").size().to_dict(), "| non-finite indicator (zero flux)", nonfinite.groupby("db").size().to_dict())
print("kept", df.groupby("db").size().to_dict())

## Sign convention check
pyrokinetics states `mode_frequency < 0` is the electron direction (comments in its CGYRO and GENE readers). Check on the
unambiguous populations: NSTX_MTM cases with tearing parity and huge `chi_e/chi_i` are MTMs and must be electron direction;
SPR-045 + M1 cases with ballooning parity and `chi_e/chi_i` ~ 1 are the KBM population.

In [ ]:
mtm_like = df[(df["T"] > 0.7) & (df.chi_ratio > 30)]
kbm_like = df[(df["T"] < 0.05) & df.chi_ratio.between(0.5, 2) & ~df.db.str.startswith("NSTX")]
print(f"MTM-like (T>0.7, chi_e/chi_i>30): {len(mtm_like)} cases, omega<0 (electron) in {(mtm_like.omega < 0).mean():.3f}")
print(f"KBM-like (T<0.05, chi 0.5-2): {len(kbm_like)} cases, omega>0 (ion) in {(kbm_like.omega > 0).mean():.3f}")

## Indicator distributions and pairwise scatters (cutoffs from the module drawn as lines)

In [ ]:
col = dict(zip(df.db.unique(), plt.cm.tab20.colors))
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
bins = {"T": np.logspace(-6, 0, 40), "omega": np.linspace(-5, 5, 50), "chi_ratio": np.logspace(-2, 3.5, 40)}
for a, (k, b) in zip(ax, bins.items()):
    for db, g in df.groupby("db"):
        a.hist(np.clip(g[k], b[0], b[-1]), bins=b, histtype="step", label=db, color=col[db])
    a.set_xlabel(k)
    if k in ("T", "chi_ratio"): a.set_xscale("log")
ax[0].axvline(mc.T_BAL, c="k", ls="--"); ax[0].axvline(mc.T_TEAR, c="k", ls=":")
ax[1].axvline(0, c="k", ls="--")
ax[2].axvspan(*mc.CHI_KBM, color="grey", alpha=0.2)
ax[0].legend(fontsize=5)
fig.savefig(plot_dir / "distributions.png")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
pairs = [("T", "chi_ratio"), ("omega", "chi_ratio"), ("omega", "T")]
for a, (x, y) in zip(ax.flat, pairs):
    for db, g in df.groupby("db"):
        a.scatter(g[x], g[y], s=6, alpha=0.5, label=db, c=col[db])
    a.set_xlabel(x); a.set_ylabel(y)
    if x == "T": a.set_xscale("symlog", linthresh=1e-3)
    if y == "chi_ratio": a.set_yscale("symlog", linthresh=0.1); a.axhspan(*mc.CHI_KBM, color="grey", alpha=0.15)
    if y == "T": a.set_yscale("symlog", linthresh=1e-3); a.axhline(mc.T_BAL, c="k", ls="--"); a.axhline(mc.T_TEAR, c="k", ls=":")
    if x == "T": a.axvline(mc.T_BAL, c="k", ls="--"); a.axvline(mc.T_TEAR, c="k", ls=":")
    if x == "omega": a.axvline(0, c="k", ls="--")
ax[0].legend(fontsize=5)
fig.savefig(plot_dir / "pairwise_scatter.png")

## Consensus labels and single-indicator disagreement

In [ ]:
df["label"] = [mc.classify(r) for r in df.to_dict("records")]
frac = df.groupby("db").label.value_counts().unstack(fill_value=0).reindex(columns=["KBM", "MTM", "mixed"], fill_value=0)
counts = frac.assign(total=frac.sum(axis=1), loaded=raw.groupby("db").size(), load_failed=failed.groupby("db").size(),
                     stable=stable.groupby("db").size(), zero_flux=nonfinite.groupby("db").size()).fillna(0).astype(int)
print(counts); counts.to_csv(plot_dir / "counts.csv")
single = pd.DataFrame({
    "omega": np.where(df.omega > 0, "KBM", "MTM"),
    "T": np.select([df["T"] < mc.T_BAL, df["T"] > mc.T_TEAR], ["KBM", "MTM"], "mixed"),
    "chi": np.where(df.chi_ratio.between(*mc.CHI_KBM), "KBM", "mixed")})
rows = {k: df.assign(s=v).groupby("db").apply(lambda g: ((g.s != "mixed") & (g.s != g.label)).sum()) for k, v in single.items()}
print("cases where the single indicator gives KBM/MTM but the consensus label differs:")
print(pd.DataFrame(rows))


## Comparison with Kotschenreuther 2019, Table 1
Table 1 (A): MHD-like (KBM) chi_i/chi_e ~ 1; MTM chi_i/chi_e ~ 1/10. (B): ITG/TEM chi_e/chi_i 1/4 to 1.
Shown as chi_e/chi_i (inverse of the table's chi_i/chi_e). Boxes are the 25-75% range of our labelled cases.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
groups = [df[df.label == l].chi_ratio for l in ("KBM", "MTM", "mixed")]
ax.boxplot([g.clip(-50, 500) for g in groups], tick_labels=[f"{l}\n(n={len(g)})" for l, g in zip(("KBM", "MTM", "mixed"), groups)], showfliers=False)
ax.scatter([1, 2], [1, 10], marker="*", s=200, c="gold", edgecolor="k", label="Kotschenreuther T1", zorder=5)
ax.set_ylabel("chi_e/chi_i"); ax.set_yscale("log"); ax.legend()
fig.savefig(plot_dir / "kotschenreuther_comparison.png")
print(df.groupby("label")[["chi_ratio", "T", "omega"]].quantile([.25, .5, .75]).round(3))

## Spot check: 5 random labelled cases per class against GS2 |A_par| and |phi|
Seeded random draw (seed above). Eigenfunctions are from the same gk_output, normalised to their own maximum.

In [ ]:
rng = np.random.default_rng(seed)
for lab in ("KBM", "MTM", "mixed"):
    pick = df[(df.label == lab) & df.db.isin(["NSTX_MTM n1000", "SPR-045", "M1"])]
    pick = pick.iloc[rng.choice(len(pick), min(n_spot, len(pick)), replace=False)]
    fig, ax = plt.subplots(2, len(pick), figsize=(3.2 * len(pick), 5), squeeze=False)
    for j, r in enumerate(pick.itertuples()):
        ds = cubes[r.db]
        s = int(np.flatnonzero(ds.sample_name.values == r.run)[0])
        for i, f in enumerate(("apar", "phi")):
            v = ds[f].isel(sample=s).squeeze().pint.dequantify()
            ax[i, j].plot(v.theta, np.abs(v) / np.abs(v).max())
            ax[i, j].set_xlabel("theta")
        ax[0, j].set_title(f"{r.db} {r.run}\nT={r.T:.2f} w={r.omega:.2f}\nchi={r.chi_ratio:.2f}", fontsize=7)
    ax[0, 0].set_ylabel("|A_par|"); ax[1, 0].set_ylabel("|phi|")
    fig.suptitle(f"consensus label: {lab}")
    fig.savefig(plot_dir / f"spotcheck_{lab}.png")